# Felsentein notebook. functions and small example for prrof of concept and testing

In [ ]:
import tskit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, SVG
from scipy.linalg import expm

In [ ]:
# this is a tree with ACTG
tables = tskit.TableCollection(sequence_length=1)

# Nodes 0–4 are samples; nodes 5–8 are ancestors.
for u, time in enumerate([0, 0, 0, 0, 0, 1, 1, 2, 3]):
    tables.nodes.add_row(
        flags=tskit.NODE_IS_SAMPLE if u < 5 else 0,
        time=time,
    )

for parent, child in [
    (5, 0), (5, 1),
    (6, 3), (6, 4),
    (7, 2), (7, 6),
    (8, 5), (8, 7),
]:
    tables.edges.add_row(left=0, right=1, parent=parent, child=child)

# One site, assuming ancestral G.
site = tables.sites.add_row(position=0.5, ancestral_state="G")
for node, allele, time in [
    (0, "C", 0.5),
    (1, "A", 0.5),
    (2, "C", 1.0),
    (3, "A", 0.5),
]:
    tables.mutations.add_row(
        site=site, node=node, derived_state=allele, time=time
    )

tables.sort()
ts = tables.tree_sequence()
tree = ts.first()

labels = {u: str(u) for u in range(9)}
labels.update({0: "0: C", 1: "1: A", 2: "2: C", 3: "3: A", 4: "4: G"})

display(SVG(tree.draw_svg(
    size=(740, 440),
    time_scale="time",
    node_labels=labels,
    mutation_labels={m.id: str(m.id) for m in ts.mutations()},
    style=".mut .sym {stroke: red;} .mut .lab {fill: red;}",
)))

In [ ]:
# this is a tree with 0/1
tables = tskit.TableCollection(sequence_length=1)

# Nodes 0–4 are samples; nodes 5–8 are ancestors.
for u, time in enumerate([0, 0, 0, 0, 0, 1, 1, 2, 3]):
    tables.nodes.add_row(
        flags=tskit.NODE_IS_SAMPLE if u < 5 else 0,
        time=time,
    )

for parent, child in [
    (5, 0), (5, 1),
    (6, 3), (6, 4),
    (7, 2), (7, 6),
    (8, 5), (8, 7),
]:
    tables.edges.add_row(left=0, right=1, parent=parent, child=child)

# One site, ancestral state 0; every mutation is 0 -> 1.
site = tables.sites.add_row(position=0.5, ancestral_state="0")
for node, allele, time in [
    (0, "0", 0.5),
    (1, "0", 0.5),
    (2, "1", 1.0),
    (3, "1", 0.5),
]:
    tables.mutations.add_row(
        site=site, node=node, derived_state=allele, time=time
    )

tables.sort()
ts = tables.tree_sequence()
tree_binary = ts.first()

labels = {u: str(u) for u in range(9)}
labels.update({0: "0: 0", 1: "1: 0", 2: "2: 1", 3: "3: 1", 4: "4: 0"})

display(SVG(tree_binary.draw_svg(
    size=(740, 440),
    time_scale="time",
    node_labels=labels,
    mutation_labels={m.id: str(m.id) for m in ts.mutations()},
    style=".mut .sym {stroke: red;} .mut .lab {fill: red;}",
)))

In [ ]:
#jukes cantor model probabilities
def calculate_jukes_cantor_prob(t_branch):
    """Calculate the transition matrix for a Jukes-Cantor model."""
    p_same = 1/4 + 3/4 * np.exp(-4*t_branch/3)
    p_diff = 1/4 - 1/4 * np.exp(-4*t_branch/3)

    P = np.full((4, 4), p_diff)
    np.fill_diagonal(P, p_same)

    return P

In [ ]:
# felsenstien jukes cantor w ACTG
def felsenstein_backwards_jc(tree, tip_states):
    "felsenstein algorithm to calculate the likelihood of the tree given the tip states"

    bases = ["A", "C", "G", "T"]
    base_index = {base: i for i, base in enumerate(bases)}

    likelihoods = {}  # Dictionary to store likelihoods for each node
    best = {}  # joint: probability of the best assignment below each node
    choice = {}  # joint: choice[child][i] = best state for the child if its parent is in state i


    for u in tree.nodes(order="postorder"):
            
            # set likelihoods for the tips
            if tree.is_leaf(u):
                likelihoods[u] = np.zeros(4)
                likelihoods[u][base_index[tip_states[u]]] = 1.0
                best[u] = likelihoods[u].copy()
                continue
            
            likelihoods[u] = np.ones(4)  # Initialize internal nodes with uniform likelihoods, 1 since we will multiply in the children's contributions
            best[u] = np.ones(4)

            # go through internal nodes
            for child in tree.children(u):
                branch_length = tree.branch_length(child)
                # later we can upgrade this so that we can save the transition matrix for each branch length, but for now we will just calculate it on the fly
                P = calculate_jukes_cantor_prob(branch_length)
                # sum over all possible states of the child, weighted by the transition probabilities
                child_states_sum = P @ likelihoods[child]
                # multiply the parent's likelihoods by the sum of the child's states
                likelihoods[u] *= child_states_sum
                # print("likelihoods:", likelihoods[u])


                # joint: same products, but keep the largest
                # scores[i, j] = P[i, j] * best[child][j]   (parent in state i, child in state j)
                scores = P * best[child]
                # for each parent state i, remember which child state j was best
                choice[child] = scores.argmax(axis=1)
                # multiply the parent's best by the child's best score
                best[u] *= scores.max(axis=1)

    # joint: the root picks its own state (jc: every base has frequency 1/4)
    root = tree.root
    root_scores = 0.25 * best[root]
    path_probability = root_scores.max()

    # joint: traceback from the root to the leaves, reading off the stored choices
    states = {root: bases[root_scores.argmax()]}
    num_mutations = 0
    for u in tree.nodes(order="preorder"):
        if u != root:
            parent_index = base_index[states[tree.parent(u)]]
            states[u] = bases[choice[u][parent_index]]
            if states[u] !=states[tree.parent(u)]:
                num_mutations +=1


            
    return likelihoods, states, path_probability, num_mutations


In [ ]:
# try the ACTG example
likelihoods, states, path_probability, num_mutations = felsenstein_backwards_jc(tree, tip_states={0: "C", 1: "A", 2: "C", 3: "A", 4: "G"})


In [ ]:
# print out things
print("Likelihoods at each node:"   )
for u in tree.nodes():
    print(f"Node {u}: {likelihoods[u]}")
print(f"Most likely states at each node: {states}")
print(f"Path probability: {path_probability}")
print(f"number of mutations: {num_mutations}")


In [ ]:
# just for the sake of visualization, let's see the tree again
display(SVG(tree.draw_svg(
    size=(740, 440),
    time_scale="time",
    node_labels=labels,
    mutation_labels={m.id: str(m.id) for m in ts.mutations()},
    style=".mut .sym {stroke: red;} .mut .lab {fill: red;}",
)))

In [ ]:
# now felsenstein with 0, 1 states. we make a general felsenstein function and apply it to the same example, but with 0, 1 states instead of A, C, G, T.

In [ ]:
# general felsenstein function that can handle any number of states, including 0/1 and ACTG. We can also impose the root to be the ancestral state
def felsenstein_backwards_general(tree, tip_states, bases=(0, 1), mu= None, root_state = None):
    "felsenstein algorithm to calculate the likelihood of the tree given the tip states"

    num_states = len(bases) 
    base_index = {b: i for i, b in enumerate(bases)}   # label -> row/column in P
    if mu is None:
        mu = 1 / (num_states - 1)
    Q = calculate_transition_matrix(mu, num_states)

    likelihoods = {}  # Dictionary to store likelihoods for each node
    best = {}  # joint: probability of the best assignment below each node
    choice = {}  # joint: choice[child][i] = best state for the child if its parent is in state i


    for u in tree.nodes(order="postorder"):
        #set likelihoods for the tips
        if tree.is_leaf(u):
            likelihoods[u] = np.zeros(num_states)
            likelihoods[u][base_index[tip_states[u]]] = 1.0
            best[u] = likelihoods[u].copy()
            continue
        
        likelihoods[u] = np.ones(num_states)  # Initialize internal nodes with uniform likelihoods, 1 since we will multiply in the children's contributions
        best[u] = np.ones(num_states)
                
        for child in tree.children(u):
            branch_length = tree.branch_length(child)
            # later we can upgrade this so that we can save the transition matrix for each branch length, but for now we will just calculate it on the fly
            P = probability_matrix(Q, branch_length)
            # sum over all possible states of the child, weighted by the transition probabilities
            child_states_sum = P @ likelihoods[child]
            # multiply the parent's likelihoods by the sum of the child's states
            likelihoods[u] *= child_states_sum
            # print("likelihoods:", likelihoods[u])


            # joint: same products, but keep the largest
            # scores[i, j] = P[i, j] * best[child][j]   (parent in state i, child in state j)
            scores = P * best[child]
            # for each parent state i, remember which child state j was best
            choice[child] = scores.argmax(axis=1)
            # multiply the parent's best by the child's best score
            best[u] *= scores.max(axis=1)

    # joint: the root picks its own state, or is fixed to root_state
    root = tree.root
    if root_state is None:
        root_scores = (1/num_states) * best[root]
    else:
        root_scores = np.zeros(num_states)
        root_scores[base_index[root_state]] = best[root][base_index[root_state]]
    path_probability = root_scores.max()

    # joint: traceback from the root to the leaves, reading off the stored choices
    states = {root: bases[root_scores.argmax()]}
    num_mutations = 0
    for u in tree.nodes(order="preorder"):
        if u != root:
            parent_index = base_index[states[tree.parent(u)]]
            states[u] = bases[choice[u][parent_index]]
            if states[u] !=states[tree.parent(u)]:
                num_mutations +=1


            
    return likelihoods, states, path_probability, num_mutations


In [ ]:
# explicit calculation of the transition matrix for a continuous-time Markov chain
def calculate_transition_matrix(mu, num_states = 2):
    """Calculate the transition matrix."""
    Q = np.full((num_states, num_states), mu)
    np.fill_diagonal(Q, -(num_states - 1) * mu)
    return Q

def probability_matrix(Q, t):
    return expm(Q * t)

In [ ]:
# check that the general still works for the ACTG example
likelihoods, states, path_probability, num_mutations = felsenstein_backwards_general(tree, tip_states={0: "C", 1: "A", 2: "C", 3: "A", 4: "G"}, bases=["A", "C", "G", "T"], root_state = "G")

In [ ]:
# just for the sake of visualization, let's see the tree again
display(SVG(tree.draw_svg(
    size=(740, 440),
    time_scale="time",
    node_labels=labels,
    mutation_labels={m.id: str(m.id) for m in ts.mutations()},
    style=".mut .sym {stroke: red;} .mut .lab {fill: red;}",
)))

In [ ]:
# print stuff out
print("Likelihoods at each node:"   )
for u in tree.nodes():
    print(f"Node {u}: {likelihoods[u]}")
print(f"Most likely states at each node: {states}")
print(f"Path probability: {path_probability}")
print(f"number of times the branch has to get a mutation: ", num_mutations)

In [ ]:
# check that the general still works for the 0/1 example
likelihoods, states, path_probability, num_mutations = felsenstein_backwards_general(tree_binary,  tip_states={0:0, 1:0, 2:1, 3:1, 4:0}, bases=[0, 1], root_state= 1)

In [ ]:
# print out things
print("Likelihoods at each node:"   )
for u in tree.nodes():
    print(f"Node {u}: {likelihoods[u]}")
print(f"Most likely states at each node: {states}")
print(f"Path probability: {path_probability}")
print("Number of times there has to be a mutation: ", num_mutations)

In [ ]:
# look at the tree again
display(SVG(tree_binary.draw_svg(
    size=(740, 440),
    time_scale="time",
    node_labels=labels,
    # mutation_labels={m.id: str(m.id) for m in ts.mutations()},
    style=".mut .sym {stroke: red;} .mut .lab {fill: red;}",
)))